# Lab01 — Cotizaciones Óptimas de un Formador de Mercado

Este notebook **solo importa funciones de `src/` y genera gráficas**.
Toda la lógica de modelo y simulación vive en `src/model.py` y `src/simulation.py`.

In [ ]:
import sys
import pathlib

import numpy as np

ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.model import (
    BASE_PI_I,
    BASE_PI_L,
    BASE_S0,
    LIQUIDITY_INTERCEPT,
    LIQUIDITY_SLOPE,
    execution_prob,
    expected_loss_ask,
    expected_loss_bid,
    optimize_quotes,
    price_pdf,
)
from src.simulation import monte_carlo, run_regime
from src.plots import (
    plot_cumulative_pnl,
    plot_execution_probability,
    plot_inventory_paths,
    plot_loss_functions,
    plot_monte_carlo_totals,
    plot_pnl_distributions,
    plot_price_distribution,
    plot_sensitivity_pi_I,
)

SEED = 42
N_TRADES = 10_000
MC_RUNS = 1_000
MC_TRADES = 1_000
SENSITIVITY_PI_I = [0.1, 0.4, 0.7]

np.random.seed(SEED)

## 1. Distribución del precio verdadero $f(P)$

In [ ]:
fig = plot_price_distribution(price_pdf, BASE_S0)
fig

La distribución Erlang(K=60, λ=3) tiene media K/λ = 20, muy cercana a
S₀ = 19.90. La mayoría de la masa se concentra entre 15 y 25, lo que
significa que el precio verdadero rara vez se desvía mucho de la
estimación del dealer. Esto es importante porque determina qué tan
frecuentemente un trader informado encontrará oportunidad de operar.

## 2. Optimización de Bid y Ask (caso base)

In [ ]:
result = optimize_quotes(S0=BASE_S0, pi_I=BASE_PI_I, pi_L=BASE_PI_L)
print(f"Bid optimo:        {result['bid']:.2f}")
print(f"Ask optimo:        {result['ask']:.2f}")
print(f"Spread optimo:     {result['spread']:.2f}")
print(f"Utilidad esperada: {result['expected_utility']:.2f}")

El spread óptimo de 6.98 es amplio: el dealer cotiza Bid=16.45 y
Ask=23.43, bastante lejos de S₀=19.90. Esto refleja que con un 40% de
traders informados, el dealer necesita protegerse significativamente.
La utilidad esperada de 0.84 por trade es lo máximo que puede obtener
bajo estas condiciones.

## 3. Pérdida esperada frente a traders informados, por lado

In [ ]:
A_range = np.linspace(BASE_S0, BASE_S0 + 10, 100)
B_range = np.linspace(0.5, BASE_S0, 100)
loss_ask = [expected_loss_ask(A) for A in A_range]
loss_bid = [expected_loss_bid(B) for B in B_range]

fig = plot_loss_functions(A_range, loss_ask, B_range, loss_bid)
fig

Ambas curvas de pérdida son decrecientes: conforme el Ask sube (o el
Bid baja), menos traders informados encuentran rentable operar, y la
pérdida esperada del dealer cae. Esto confirma que ampliar el spread
reduce la selección adversa, pero a costa de perder operaciones con
traders de liquidez.

## 4. Probabilidad de ejecución vs. spread respecto a $S_0$

In [ ]:
zero_spread = LIQUIDITY_INTERCEPT / LIQUIDITY_SLOPE
spread_range = np.linspace(0, zero_spread + 1, 200)
exec_prob = execution_prob(spread_range)

fig = plot_execution_probability(spread_range, exec_prob, zero_spread)
fig

La probabilidad de ejecución de un trader de liquidez decae linealmente
con el spread y llega a cero en s = 0.50/0.08 = 6.25. Esto significa
que si el dealer pone su cotización a más de 6.25 de S₀, ningún trader
de liquidez operará. El spread óptimo (6.98) supera ligeramente este
punto, lo que implica que la ganancia por liquidez en el óptimo ya es
muy pequeña.

## 5. Simulación de 10,000 trades bajo tres regímenes

In [ ]:
regimes = {
    "Optimo": (result["bid"], result["ask"]),
    "Estrecho": (19.75, 20.05),
    "Amplio": (18.40, 21.40),
}

pnl_by_regime = {}
inventory_by_regime = {}
for name, (bid, ask) in regimes.items():
    stats_regime = run_regime(N_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    pnl_by_regime[name] = stats_regime["pnl"]
    inventory_by_regime[name] = stats_regime["inventory_path"]
    print(
        f"{name:10s} Bid={bid:6.2f} Ask={ask:6.2f}  "
        f"PnL total={stats_regime['total_pnl']:10.2f}  PnL medio={stats_regime['mean_pnl']:7.4f}  "
        f"Inv final={stats_regime['final_inventory']:8.0f}  |Inv| max={stats_regime['max_abs_inventory']:8.0f}"
    )

### PnL acumulado

In [ ]:
fig = plot_cumulative_pnl(pnl_by_regime)
fig

El régimen Estrecho pierde consistentemente: su curva de PnL acumulado
cae de forma sostenida porque los traders informados lo explotan en
casi cada trade. El Óptimo crece de forma estable y el Amplio crece
pero más lento, porque aunque se protege de los informados, también
deja pasar muchas oportunidades de ganancia con los de liquidez.

### Distribución del PnL por trade

In [ ]:
fig = plot_pnl_distributions(pnl_by_regime)
fig

### Inventario acumulado

In [ ]:
fig = plot_inventory_paths(inventory_by_regime)
fig

El Estrecho acumula el mayor desbalance de inventario porque casi todos
los trades se ejecutan (la probabilidad de ejecución es alta cuando el
spread es chico). Esto expone al dealer a riesgo de precio: si se
queda con inventario largo y el precio baja, pierde dinero. El modelo
no captura este riesgo porque evalúa cada trade de forma independiente.

## 6. Monte Carlo: 1,000 corridas de 1,000 trades

In [ ]:
totals_by_regime = {}
for name, (bid, ask) in regimes.items():
    totals = monte_carlo(MC_RUNS, MC_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    totals_by_regime[name] = totals
    prob_loss = float((totals < 0).mean())
    print(
        f"{name:10s} media={totals.mean():10.2f}  "
        f"std={totals.std():8.2f}  "
        f"P(perdida)={prob_loss:.4f}"
    )

In [ ]:
fig = plot_monte_carlo_totals(totals_by_regime)
fig

El histograma confirma la separación completa entre regímenes: las
distribuciones de PnL final no se traslapan. El Estrecho tiene
probabilidad de pérdida del 100% (todas las corridas terminan en
negativo), mientras que el Óptimo y el Amplio nunca pierden en 1,000
corridas. La desviación estándar es similar en los tres casos (~45),
lo que indica que la volatilidad del PnL depende más del número de
trades que del régimen.

## 7. Análisis de sensibilidad: spread óptimo vs. $\pi_I$

In [ ]:
sensitivity_spreads = []
for pi_I in SENSITIVITY_PI_I:
    pi_L = 1.0 - pi_I
    result_pi_I = optimize_quotes(S0=BASE_S0, pi_I=pi_I, pi_L=pi_L)
    sensitivity_spreads.append(result_pi_I["spread"])
    print(f"pi_I={pi_I:4.2f}  Bid={result_pi_I['bid']:6.2f}  Ask={result_pi_I['ask']:6.2f}  Spread={result_pi_I['spread']:6.2f}")

In [ ]:
fig = plot_sensitivity_pi_I(SENSITIVITY_PI_I, sensitivity_spreads)
fig

El spread óptimo crece de forma monótona con πᵢ: de 6.40 con 10% de
informados a 7.99 con 70%. Esto coincide con la teoría de
Copeland-Galai: a mayor probabilidad de enfrentar traders informados,
el dealer necesita un spread más amplio para compensar las pérdidas
por selección adversa.